# Reading habits throughout University Years (2022-2025)

The following analysis aims to explore my reading habits throughout the last four years. In the school year 2021/2022, I enrolled at the University of Bologna as a Bachelor's student. Three years later, I decided to follow a Master's degree in DHDK. Almost five years have passed since September 2021 but for clarity I decided to create a dataset recording the books I have read for pleasure since January 2022 until December 2025. The books read recently in 2026 are not part of the dataset because they would not represent the year precisely since the list is, of course, still incomplete. 

First of all, let's take a look at the attributes of my dataset. For each of the last four years I recorded the title of the book read, the name of the author, the number of pages, its genre, the starting month of a specific reading, the end month, the start year and the end year. 
I created the dataset to perform an **EDA (exploratory data analysis)** on my reading habits which will respond to the following questions:

1. How much did I read in these specific years (2022 to 2025)? Was I consistent throughout the years?
2. What are my tastes in book genres? 
3. In which months do I read the most? Can I find some patterns?

The answer to these questions will help me understand how much I engaged with reading during the years, it will give me a clear overview of what I tend to like and return to and when I prioritized reading throughout the years. 


In [1]:
import pandas as pd
df = pd.read_csv("reading_habits.csv", sep=";", encoding="utf-8")
print(df.head())

                                    Title          Author  Number of pages  \
0                      Un polpo alla gola     Zerocalcare              204   
1     Tempo. Il sogno di uccidere Chrónos   Guido Tonelli              192   
2                             Frida Kahlo     Rauda Jamis              265   
3  L'insostenibile leggerezza dell'essere  Milan  Kundera              352   
4                       Il giovane Holden  J. D. Salinger              251   

               Genre Start month End month  Start year  End year  
0      Graphic novel     January   January        2022      2022  
1        Non-fiction    February  February        2022      2022  
2        Non-fiction       March     March        2022      2022  
3  Literary fiction        April     April        2022      2022  
4  Literary fiction         May        May        2022      2022  


The cell below assigns a fixed order to the months, since I want to visualize them in a ordered sequence. To manage books I started reading in a month and ended in a different one (e.g. "Una vita come tante", started in May 2023 and ended in August 2023), I decided to divide the number of pages equally by the number of months it took for me to finish it.  

In [2]:
months_order = ["January", "February", "March", "April", "May", "June", 
                "July", "August", "September", "October", "November", "December"]

month_to_num = {m: i+1 for i, m in enumerate(months_order)}
num_to_month = {i+1: m for i, m in enumerate(months_order)}

row_list = []

for _, row in df.iterrows():
    m_start = month_to_num[row['Start month'].strip()]
    m_end = month_to_num[row['End month'].strip()]
    
    start_abs = m_start + (int(row['Start year']) * 12)
    end_abs = m_end + (int(row['End year']) * 12)
    
    duration = end_abs - start_abs + 1
    if duration <= 0: duration = 1 

    pages_per_month = row['Number of pages'] / duration

    for m_abs in range(start_abs, end_abs + 1):
        y = (m_abs - 1) // 12
        m_num = (m_abs - 1) % 12 + 1

        row_list.append({
            'Title': row['Title'],
            'Genre': row['Genre'].strip(),
            'Year': y,
            'Month': num_to_month[m_num],
            'Pages': round(pages_per_month, 1)
        })

df_full = pd.DataFrame(row_list)

unique_years = sorted(df_full['Year'].unique())
unique_genres = sorted(df_full['Genre'].unique())
n_years = len(unique_years)

### First Sub-question Visualization
**1. How much did I read in these specific years (2022 to 2025)? Was I consistent throughout the years?**

To answer this specific preliminary question, I compared the annual number of pages through a line chart. This visualization highlights the overall trend during the last four years of university, where the data points represent the total number of pages read each year. The chart shows peaks and lows in my reading engagement over the long term.


In [3]:
import plotly.express as px

yearly_trend = df_full.groupby('Year')['Pages'].sum().reset_index()
y_values = list(range(0, int(yearly_trend['Pages'].max()) + 1000, 100))

line_chart = px.line(yearly_trend, 
                     x="Year",
                     y="Pages",
                     markers=True,
                     line_shape="spline",
                    )

line_chart.update_traces(
                        line_color='#FF8C00',    
                        line_width=4,          
                        marker=dict(size=12, color='#2C3E50', line=dict(width=2, color='white')), 
                        textposition="top center",
                        textfont=dict(size=14, family="Arial"),
                        hoverlabel=dict(
                        bgcolor="#F5DEB3",      
                        bordercolor="black",    
                        font_size=14,             
                        font_family="Courier New, monospace",
                        font_color="black"
                    ),
                        hovertemplate="<b>Year %{x}</b><br>Total Pages: <b>%{y}</b><extra></extra>"
                        )

line_chart.update_layout(
    template="plotly_white",   
    title=dict(
        text="<b>Annual reading trend throughout University years (2022-2025)</b>", 
        x=0.5,            
        font=dict(size=24, family="Courier New, monospace")
    ),
    xaxis=dict(
        title=dict(
            text="<b>Years</b>", 
            font=dict(size=18, family="Courier New, monospace") 
        ),
        tickmode='array',
        tickvals=unique_years, 
        ticktext=[f"<b>{year}</b>" for year in unique_years],
        tickfont=dict(size=18, family="Courier New, monospace"), 
        showgrid=False,        
        linecolor='black'
    ),
    yaxis=dict(
        title=dict(
            text="<b>Total Pages</b>", 
            font=dict(size=18, family="Courier New, monospace")
        ),
        tickmode='array',
        tickvals=y_values,
        ticktext=[f"<b>{val}</b>" for val in y_values],
        tickfont=dict(size=16, family="Courier New, monospace"),
        showgrid=False,        
        linecolor='black',
        fixedrange=True
    ),
    width=1000,
    height=500,
    autosize=False,
    margin=dict(t=100)
)

line_chart.update_layout(
    plot_bgcolor='#F5F5F0', 
    paper_bgcolor='#F5F5F0', 
)

line_chart.update_xaxes(fixedrange=True)
line_chart.update_yaxes(fixedrange=True)

line_chart.show()

### Second Sub-question Visualization

**2. What are my tastes in book genres?** 

To answer this question, I analyzed the distribution of the seven genres recorded in my dataset: Graphic novels, non-fiction, literary fiction, romance, sci-fi books and drama. Horror, mystery and thriller books were grouped in one single category due to their stylistic similarity. Even though I am aware of what my favourite genres are, analyzing the annual percentage provides a clearer overview of what I tend to read the most for pleasure and helps me identify shifts in my tastes. 
A pie chart is the ideal method to visualize how each genre weighs on the annual total, providing a clear insight on my preferences.

In [4]:
color_map = {
    'Literary fiction': '#FF8C00',          
    'Non-fiction': '#708090',              
    'Horror/Mystery/Thriller': '#191970',   
    'Graphic novel': '#FFFF00',          
    'Drama': '#DDA0DD',                  
    'Romance': '#FF69B4',                 
    'Sci-Fi': '#00FF00'                    
}

df_genres_sum = df_full.groupby(['Year', 'Genre'])['Pages'].sum().reset_index()

df_proportions = pd.merge(
    df_genres_sum, 
    yearly_trend.rename(columns={'Pages': 'Total_Year_Pages'}), 
    on='Year'
)

df_proportions['Percentage'] = (df_proportions['Pages'] / df_proportions['Total_Year_Pages'] * 100).round(1)

In [30]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

subplot_titles = [f"<b>Year {year}</b>" for year in unique_years]

fig_pies = make_subplots(
    rows=2, cols=2,
    subplot_titles=subplot_titles,
    specs=[[{'type': 'domain'}, {'type': 'domain'}],
           [{'type': 'domain'}, {'type': 'domain'}]],
    vertical_spacing=0.12,   
    horizontal_spacing=0.1   
)

for i, year in enumerate(unique_years):
    year_data = df_full[df_full['Year'] == year].groupby('Genre')['Pages'].sum().reset_index()
 
    year_data['Genre'] = pd.Categorical(year_data['Genre'], ordered=True)
    year_data = year_data.sort_values('Genre')
    
    row = (i // 2) + 1
    col = (i % 2) + 1
    
    fig_pies.add_trace(
        go.Pie(
            labels=year_data['Genre'],
            values=year_data['Pages'],
            name=str(year),
            marker=dict(colors=[color_map.get(g, "grey") for g in year_data['Genre']]),
            textinfo='percent',
            textfont=dict(size=18, family="Courier New, monospace", color="black"),
            hole=0, 
            pull=[0.05] * len(year_data),
            hovertemplate="<b>Genre: %{label}</b><br>%{value} Pages<br>Percentage: <b>%{percent}</b><extra></extra>"
        ),
        row=row, col=col
    )

fig_pies.update_layout(
    title_text="<b>Book genres distribution per year (2022-2025)</b>",
    title_x=0.5,
    title_font=dict(size=26, family="Courier New, monospace"),
    font=dict(family="Courier New, monospace"),
    showlegend=True,
    legend=dict(
        title="<b>Genres</b>",
        orientation="h",
        entrywidth=0.3,
        entrywidthmode="fraction",
        y=-0.08,
        x=0.6, 
        xanchor="center",
        font=dict(size=14),
        itemsizing='constant'
    ),
    width=1100,  
    height=950,  
    margin=dict(t=120, b=150, l=50, r=50), 
    hoverlabel=dict(
        bgcolor="#F5DEB3",      
        bordercolor="black",    
        font_size=14,             
        font_family="Courier New, monospace",
        font_color="black",
                
    )
)


for i in fig_pies['layout']['annotations']:
    i['font'] = dict(size=20, family="Courier New, monospace")


fig_pies.update_layout(
    plot_bgcolor='#F5F5F0', 
    paper_bgcolor='#F5F5F0', 
)

fig_pies.show(config={'staticPlot': False, 'responsive': False})

### A Comprehensive View of My Reading Habits Throughout University Years
In this final analysis I want to answer the last question:

**3. In which months do I read the most? Can I find some patterns?**

By doing so I want to integrate also the results from the previous charts in a comprehensive stacked bar chart. In the following graph it is possible to visualize the distribution of pages read in each month of the last four years. Inside the monthly bar different genres are distinguished by color. The navy blue dotted line represents the scansion of the years while the red one distinguishes the different seasons inside each year. This distinction allows ond to interpret the data considering the temporal period.
 

In [104]:
import pandas as pd
import plotly.express as px

df_full['Date'] = pd.to_datetime(df_full['Year'].astype(str) + '-' + df_full['Month'], format='%Y-%B')
df_full = df_full.sort_values('Date')
df_full['Month_Year'] = df_full['Date'].dt.strftime('%b %y')
df_full['Full_Month_Name'] = df_full['Date'].dt.strftime('%B %Y')
df_timeline = df_full.groupby(['Date', 'Month_Year', 'Full_Month_Name', 'Genre'])['Pages'].sum().reset_index()
ordered_labels = df_timeline.sort_values('Date')['Month_Year'].unique()

monthly_tot = df_timeline.groupby('Month_Year')['Pages'].transform('sum')
df_timeline['Percentage'] = (df_timeline['Pages'] / monthly_tot * 100).round(1)

df_timeline['Year_int'] = pd.to_datetime(df_timeline['Month_Year'], format='%b %y').dt.year

df_timeline = df_timeline.merge(
    df_proportions[['Year', 'Genre', 'Percentage']].rename(columns={'Percentage': 'Annual_Genre_Pct'}),
    left_on=['Year_int', 'Genre'],
    right_on=['Year', 'Genre'],
    how='left'
).drop(columns='Year')

def get_season_info(month_year_str):
    m = month_year_str.split(' ')[0]
    if m in ['Dec', 'Jan', 'Feb']: return 'WINTER', '#1f77b4'
    if m in ['Mar', 'Apr', 'May']: return 'SPRING', '#FF0033'
    if m in ['Jun', 'Jul', 'Aug']: return 'SUMMER', '#FFCC00'
    return 'AUTUMN', '#663300'

def get_month_initial_bold(month_year_str):
    return f"<b>{month_year_str[0]}</b>"

final_chart = px.bar(
    df_timeline, x="Month_Year", y="Pages", color="Genre",
    custom_data=["Full_Month_Name", "Percentage", "Genre", "Annual_Genre_Pct"],
    color_discrete_map=color_map,
    category_orders={"Month_Year": list(ordered_labels)}
)

final_chart.update_layout(
    font=dict(family="Courier New, monospace", color="#2C3E50"),
    
    xaxis=dict(
        title="",
        tickmode='array',
        tickvals=list(ordered_labels),
        ticktext=[get_month_initial_bold(m) for m in ordered_labels],
        tickangle=0,
        tickfont=dict(size=11),
        type='category'
    ),
    yaxis=dict(
        title="<b>Pages Read</b>", 
        gridcolor='lightgrey',
        tickfont=dict(color="#2C3E50"),
        title_font=dict(color="#2C3E50", size=18)
    ),
    bargap=0.25,
    
    autosize=True,
    width=1420,                 
    height=600, 
    plot_bgcolor='white',
    barmode='stack',
    
    legend=dict(
        title="<b>Genres</b>",
        orientation="h",
        entrywidth=0.3, 
        entrywidthmode="fraction", 
        y=-0.15,
        x=0.55,                 
        xanchor="center",
        font=dict(size=14),
        itemsizing='constant'
    ),
   
    margin=dict(b=110, t=80, l=60, r=40),
    hoverlabel=dict(
        bgcolor="white",
        font_size=14,
        font_family="Courier New, monospace",
        font_color="black"
    )
)

annual_stats = yearly_trend.set_index('Year')['Pages'].to_dict()
years_sorted = sorted(annual_stats.keys())
annual_delta = {}
for i, y in enumerate(years_sorted):
    if i > 0:
        prev = annual_stats[years_sorted[i-1]]
        annual_delta[y] = round((annual_stats[y] - prev) / prev * 100, 1)
    else:
        annual_delta[y] = None

for i, label in enumerate(ordered_labels):
    season_name, season_color = get_season_info(label)
    
    if i % 3 == 1: 
        final_chart.add_annotation(
            x=i, y=-0.12, yref="paper",
            text=f"<b>{season_name}</b>",
            showarrow=False, 
            font=dict(size=18, color=season_color)
        )

    if i % 3 == 0 and i > 0 and i % 12 != 0:
        final_chart.add_vline(x=i-0.5, line_width=1, line_dash="dot", line_color="red")
    
    if i % 12 == 0:
        x_pos = i - 0.5
        year_str = label.split(' ')[1] 
        year_full = int(f"20{year_str}")

        total_pages = annual_stats.get(year_full, 0)

        if i > 0:
            final_chart.add_vline(x=x_pos, line_width=2.5, line_dash="dash", line_color="#2C3E50")
        
        final_chart.add_annotation(
            x=i + 5.5, y=1.15, yref="paper",
            text=f"<b>Year {year_full}</b>",
            showarrow=False, 
            font=dict(size=20, color="#2C3E50")
        )

        delta = annual_delta.get(year_full)
        if delta is not None:
            sign = "+" if delta > 0 else ""
            delta_color = "green" if delta > 0 else "red"
        else:
            sign, delta_color = "", "#2C3E50"

        final_chart.add_annotation(
            x=i + 5.5, y=1.08, yref="paper",
            text=f"<b>{int(total_pages)}</b> total pages", 
            showarrow=False, 
            font=dict(size=15, color="#2C3E50")
        )

        if delta is not None:
            final_chart.add_annotation(
                x=i + 9.5, y=1.08, yref="paper",
                text=f"<b>{sign}{delta}%</b>",
                showarrow=False,
                font=dict(size=15, color=delta_color)
            )

final_chart.update_traces(
    hovertemplate=(
    "<b>%{customdata[0]}</b><br>"
    "<b>%{customdata[2]}</b><br>"
    "Pages: <b>%{y}</b><br>"
    "Monthly share: <b>%{customdata[1]}%</b><br>"
    "Annual share: <b>%{customdata[3]}%</b>"
    "<extra></extra>"
),
    hoverlabel=dict(
        bgcolor="#F5DEB3",      
        bordercolor="black",    
        font_size=14,           
        font_family="Courier New, monospace",
        font_color="black",
    ),
)
final_chart.update_xaxes(fixedrange=True)
final_chart.update_yaxes(fixedrange=True)

final_chart.update_layout(
    plot_bgcolor='#F5F5F0', 
    paper_bgcolor='#F5F5F0', 
)


html_layout = """
<html>
<head>
    <meta charset="utf-8" />
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <style>
        html, body {{
            margin: 0;
            padding: 0;
            width: 100%;
            background-color: #F5F5F0;
            overflow_x: auto; 
            overflow_y: auto;
        }}
        
        .main-wrapper {{
            display: flex; 
            flex-direction: column; 
            align-items: center; 
            width: 100%; 
            min-width: 1420px;
            padding: 25px 15px; 
            box-sizing: border-box;
        }}

        .text-header {{
            text-align: center; 
            width: 1420px; 
            margin-bottom: 25px;
            box-sizing: border-box;
        }}

        .chart-container {{
            width: 1420px;
            margin: 0 auto;
            box-sizing: border-box;
        }}
    </style>
</head>
<body>
    <div class="main-wrapper">
        
        <div class="text-header">
            <h1 style="font-family: 'Courier New', monospace; font-size: 28px; color: #2C3E50; font-weight: bold; margin: 0 0 25px 0;">
                My Reading Habits Throughout University Years (2022-2025)
            </h1>

            <div style="font-family: 'Courier New', monospace; font-size: 16px; color: #2C3E50; font-weight: 500; text-align: center; line-height: 1.4; margin: 0; padding-right: 100px; padding-left: 100px;">
                An interactive <b><i>stacked bar chart</i></b> presenting an in-depth analysis of personal reading habits throughout university years.
                The data reveals a progressive <span style="color: green;"><b><i>increase</b></span> in annual reading volumes, it highlights distinct <b><i>seasonal fluctuations</i></b>, characterized by peaks during spring and summer
                and lows in autumn and winter. Finally, the visualization proves a preference for <span style="color: #FF8C00;"><b><i>Literary Fiction</b></span> which remains a constant genre across the entire timeframe.
            </div>
        </div>

        <div class="chart-container">
            {plot_div}
        </div>
        
    </div>
</body>
</html>
"""

plot_div = final_chart.to_html(full_html=False, include_plotlyjs='cdn', config={'responsive': True})


final_chart.update_layout(
    margin=dict(t=140), 
    title=dict(
        text="<b>My Reading Habits Throughout University Years (2022-2025)</b>",
        x=0.5,
        y=0.93,     
        xanchor="center",
        yanchor="top",      
        font=dict(size=22, color="#2C3E50")
    )
)

final_chart.show()

with open("My_Reading_Habits.html", "w", encoding="utf-8") as f:
    f.write(html_layout.format(plot_div=plot_div))

## Conclusions

By taking a look at this final chart it's possible to answer the three sub-questions defined above. The comparison between the number of pages read annually confirms how I am trying to gradually dedicate more time to reading for pleasure. Despite that, the chart shows a drop of -10% in the volume of pages read in 2023. That was an intense year characterized by many changes and moments of uncertainty. I probably did not have enough energy to fully engage in reading during my free time. 

The genre blocks of the stacked bar chart show an undisputed preference for Literary fiction, followed by a major interest in the Horror/Mystery/Thriller category. Other genres only appear during specific periods and then disappear. Among these are romance, which can only be found in Spring 2022, Sci-fi, in the middle of 2024 and Drama, at the beginning of the year 2025. These outliers can be attributed to my periodical desire to step out of my comfort zone. Despite these brief shifts, I always return to my favorite genres. 

The wave trend highlights the inconsistency of the volume of pages read throughout the span of a year. This trend tends to repeat itself, demonstrating that it is possible to find interesting patterns that match specific periods of the year. When focusing on the seasons a clear pattern emerges. During winter and autum I read significantly less than during spring and summer. I can attribute the winter breaks from reading to the exams sessions while the autumn drops can be linked to returning from the summer holidays and shifting my focus toward practical and organizational matters. I read the least during Autumn 2025, the period that corresponds to my Erasmus+ exchange. This proves that when I am focused on many other activities, reading temporarily ceases to be one of my priorities. 

The period characterized by the highest reading activity was Spring 2025. This peak marks the moment I discovered the Elena Ferrante's "L'amica geniale" saga. I remember how much I loved each book and how I spent most of my free time reading them. 

Finally, I can state with certainty that I prefer reading during spring and summer, likely because the days are longer and I find more stimulus to dedicate more time to this activity. Generally, I try to keep a constant reading habit due to a personal rule that motivates me, which consists in finishing one book per month. In a world that privileges speed, short contents and mindless scrolling, I try to resist temptation at least for 30 minutes a day and dedicate that time to reading. 

### Afterword

In this notebook I tried to analyze my reading habits by looking retrospectively at the books I collected in the last four years in my library, trying to place them perfectly into a timeline. Inevitably, I relied on my memory to gather these data. In a professional data science environment, this method would not be accepted. A data analyst should not collect data retrospectively by trying to remeber them. But instead, they should start gathering data before even planning their visualization.

For the sake of simplifying the analysis, I had to model some of my personal data to facilitate the visualization. One of the limitations that arises from the preliminary calculations I made, is that I divided the pages of the books equally by the number of months it took me to read them. This is a huge approximation that does not reflect reality. In my experience, I tend to read a large number of pages all at once if I am really interested in a book. It is not rare that I start reading a book slowly because the beginning does not particulary excite me and then the plot twist arrives and I read more than an half of it in less than three days.

It is important, when looking at this final chart, to consider this limitation and to interpret the data accordingly, keeping in mind that the visualization does not reflect reality 100% but is just as faithfull a representation as possible, based on what I remember from the last four years of my reading journey. 
